# Day 3 EDA Analysis

This notebook explores cleaned mutual fund data from `data/processed/` and references exported PNG charts in `reports/charts/day3/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
CHART_DIR = BASE_DIR / "reports" / "charts" / "day3"

sns.set_theme(style="whitegrid")

In [ ]:
fund = pd.read_csv(PROCESSED_DIR / "01_fund_master_clean.csv", parse_dates=["launch_date"])
nav = pd.read_csv(PROCESSED_DIR / "02_nav_history_clean.csv", parse_dates=["date"])
aum = pd.read_csv(PROCESSED_DIR / "03_aum_by_fund_house_clean.csv", parse_dates=["date"])
sip = pd.read_csv(PROCESSED_DIR / "04_monthly_sip_inflows_clean.csv", parse_dates=["month"])
category = pd.read_csv(PROCESSED_DIR / "05_category_inflows_clean.csv", parse_dates=["month"])
folio = pd.read_csv(PROCESSED_DIR / "06_industry_folio_count_clean.csv", parse_dates=["month"])
performance = pd.read_csv(PROCESSED_DIR / "07_scheme_performance_clean.csv")
transactions = pd.read_csv(PROCESSED_DIR / "08_investor_transactions_clean.csv", parse_dates=["transaction_date"])
holdings = pd.read_csv(PROCESSED_DIR / "09_portfolio_holdings_clean.csv", parse_dates=["portfolio_date"])
benchmark = pd.read_csv(PROCESSED_DIR / "10_benchmark_indices_clean.csv", parse_dates=["date"])

{name: df.shape for name, df in {
    "fund": fund, "nav": nav, "aum": aum, "sip": sip, "category": category,
    "folio": folio, "performance": performance, "transactions": transactions,
    "holdings": holdings, "benchmark": benchmark
}.items()}

## Plotly NAV trend with event highlights

In [ ]:
nav_plot = nav.merge(fund[["amfi_code", "scheme_name"]], on="amfi_code", how="left")
fig = px.line(
    nav_plot,
    x="date",
    y="nav",
    color="scheme_name",
    title="Daily NAV Trend for All 40 Schemes, 2022-2026",
    labels={"date": "Date", "nav": "NAV", "scheme_name": "Scheme"}
)
fig.add_vrect(x0="2023-04-01", x1="2023-12-31", fillcolor="green", opacity=0.12, line_width=0)
fig.add_vrect(x0="2024-06-01", x1="2024-10-31", fillcolor="red", opacity=0.12, line_width=0)
fig.show()

![NAV trend](../reports/charts/day3/01_nav_trend_all_40_schemes.png)

## AUM growth and fund-house dominance

In [ ]:
aum_yearly = aum.assign(year=aum["date"].dt.year)
aum_yearly = aum_yearly[aum_yearly["year"].between(2022, 2025)]
sns.catplot(data=aum_yearly, x="fund_house", y="aum_lakh_crore", hue="year", kind="bar", height=6, aspect=2)
plt.xticks(rotation=45, ha="right")
plt.title("AUM Growth by Fund House, 2022-2025")
plt.show()

![AUM growth](../reports/charts/day3/03_aum_growth_by_fund_house.png)

## SIP inflow trend

In [ ]:
fig = px.line(sip, x="month", y="sip_inflow_crore", markers=True, title="Monthly SIP Inflows")
fig.add_annotation(x="2025-12-01", y=31002, text="Rs 31,002 Cr all-time high", showarrow=True)
fig.show()

![SIP trend](../reports/charts/day3/05_sip_inflow_time_series.png)

## Category inflow heatmap

In [ ]:
heat = category.assign(month_label=category["month"].dt.strftime("%Y-%m"))
heat = heat.pivot_table(index="category", columns="month_label", values="net_inflow_crore", aggfunc="sum")
plt.figure(figsize=(15, 6))
sns.heatmap(heat, cmap="YlGnBu")
plt.title("Category Net Inflow Heatmap")
plt.show()

![Category heatmap](../reports/charts/day3/07_category_inflow_heatmap.png)

## Investor demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
transactions["age_group"].value_counts().sort_index().plot(kind="pie", autopct="%1.1f%%", ax=axes[0], title="Age Group")
sip_tx = transactions[transactions["transaction_type"] == "SIP"]
sns.boxplot(data=sip_tx, x="age_group", y="amount_inr", ax=axes[1])
axes[1].set_title("SIP Amount by Age Group")
transactions["gender"].value_counts().plot(kind="pie", autopct="%1.1f%%", ax=axes[2], title="Gender Split")
plt.tight_layout()
plt.show()

![Age pie](../reports/charts/day3/09_age_group_distribution_pie.png)
![SIP age box](../reports/charts/day3/10_sip_amount_boxplot_by_age.png)
![Gender split](../reports/charts/day3/11_gender_split.png)

## Geographic distribution

In [ ]:
sip_state = transactions[transactions["transaction_type"] == "SIP"].groupby("state", as_index=False)["amount_inr"].sum()
sip_state = sip_state.sort_values("amount_inr", ascending=False)
plt.figure(figsize=(10, 7))
sns.barplot(data=sip_state, y="state", x="amount_inr")
plt.title("SIP Amount by State")
plt.show()

![SIP by state](../reports/charts/day3/12_sip_amount_by_state.png)
![City tier split](../reports/charts/day3/13_city_tier_split.png)

## Folio count growth

In [ ]:
plt.figure(figsize=(13, 5))
sns.lineplot(data=folio, x="month", y="total_folios_crore", marker="o")
plt.title("Industry Folio Count Growth")
plt.show()

![Folio growth](../reports/charts/day3/14_folio_count_growth.png)

## NAV return correlation matrix

In [ ]:
selected = performance.sort_values("aum_crore", ascending=False).head(10)["amfi_code"]
nav_selected = nav[nav["amfi_code"].isin(selected)].merge(fund[["amfi_code", "scheme_name"]], on="amfi_code")
nav_wide = nav_selected.pivot_table(index="date", columns="scheme_name", values="nav")
returns = nav_wide.pct_change(fill_method=None).dropna(how="all")
plt.figure(figsize=(12, 10))
sns.heatmap(returns.corr(), cmap="vlag", center=0)
plt.title("Daily Return Correlation Matrix")
plt.show()

![Correlation heatmap](../reports/charts/day3/15_nav_return_correlation_heatmap.png)

## Sector allocation

In [ ]:
equity_holdings = holdings.merge(fund[["amfi_code", "category"]], on="amfi_code")
equity_holdings = equity_holdings[equity_holdings["category"] == "Equity"]
equity_holdings.groupby("sector")["weight_pct"].sum().sort_values(ascending=False)

![Sector allocation](../reports/charts/day3/16_sector_allocation_donut.png)

## 10 Key EDA Findings

1. Chart 01 shows that NAV levels vary widely across schemes, so indexed views are needed before comparing growth.
2. Chart 02 shows that several schemes compounded meaningfully after the 2023 rally window.
3. Chart 03 shows SBI Mutual Fund as the key AUM leader, with the Rs 12.5L Cr dominance marker used as the strategic reference.
4. Chart 05 shows SIP inflows rising steadily and reaching the highlighted Rs 31,002 Cr high in Dec 2025.
5. Chart 07 shows category inflows are uneven by month, with concentrated bursts in selected fund categories.
6. Chart 09 shows investor age participation is distributed across multiple age bands rather than concentrated in one group.
7. Chart 10 shows SIP ticket sizes differ by age group, with outliers visible in several groups.
8. Chart 12 shows SIP contribution is geographically concentrated in the largest contributing states.
9. Chart 14 shows industry folios approximately doubled from 13.26 Cr to the highlighted 26.12 Cr milestone.
10. Chart 15 shows selected fund returns are positively correlated, meaning diversification across similar equity funds may still carry shared market risk.

## Exported Chart Files

- `01_nav_trend_all_40_schemes.png`
- `02_indexed_nav_growth.png`
- `03_aum_growth_by_fund_house.png`
- `04_latest_aum_ranking.png`
- `05_sip_inflow_time_series.png`
- `06_sip_inflow_vs_active_accounts.png`
- `07_category_inflow_heatmap.png`
- `08_total_category_inflows.png`
- `09_age_group_distribution_pie.png`
- `10_sip_amount_boxplot_by_age.png`
- `11_gender_split.png`
- `12_sip_amount_by_state.png`
- `13_city_tier_split.png`
- `14_folio_count_growth.png`
- `15_nav_return_correlation_heatmap.png`
- `16_sector_allocation_donut.png`
- `17_risk_return_scatter.png`
- `18_expense_ratio_vs_return.png`